# Modul 09: scikit-learn-Estimatoren und Pipelines

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Estimator-API, Pipelines bauen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Grundlagen  
    **Orientierungszeit:** etwa 105 bis 145 Minuten

    ## Überblick

    Sie arbeiten systematisch mit der scikit-learn-Estimator-API und bauen eine leakage-sichere Pipeline für gemischte numerische und kategoriale Daten. Parameter, gelernte Attribute, Schritte und Merkmalsnamen werden gezielt geprüft.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_09A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_09B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - scikit-learn-Versionen und kleine Beispieldatensätze prüfen.
- fit, transform, predict, Parameter und gelernte Attribute unterscheiden.
- Dummy-Baselines und lineare Modelle auf Standarddaten trainieren.
- Skalierer, Imputer und Encoder passend auf Spalten anwenden.
- ColumnTransformer und Pipeline zu einem End-to-End-Workflow verbinden.
- Pipeline-Schritte, verschachtelte Parameter und Merkmalsnamen untersuchen.

    ## Bewertete Fähigkeiten

    - Estimator-API, get_params und gelernte Attribute
- DummyClassifier, DummyRegressor, LogisticRegression und LinearRegression
- SimpleImputer, StandardScaler und OneHotEncoder
- ColumnTransformer, Pipeline, Feature-Namen und verschachtelte Parameter

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_diabetes, load_iris
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

mixed_data = pd.DataFrame(
    {
        "age": [25, 31, 48, 52, 37, np.nan, 44, 29, 61, 33, 41, 55, 27, 46, 39, 58],
        "income_k": [42, 55, 88, 95, np.nan, 63, 79, 49, 110, 58, 72, 102, 45, 85, 68, 106],
        "city": ["Berlin", "München", "Berlin", "Hamburg", "Berlin", "München", "Hamburg", "Berlin", "Köln", "München", "Hamburg", "Köln", "Berlin", "Hamburg", "München", "Köln"],
        "plan": ["Basis", "Basis", "Plus", "Plus", "Basis", "Plus", "Plus", "Basis", "Premium", "Basis", "Plus", "Premium", "Basis", "Plus", "Plus", "Premium"],
        "renewed": [0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1],
    }
)
print("scikit-learn:", sklearn.__version__)
print(mixed_data.head())

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Estimator-API an Klassifikation und Regression prüfen

    Bearbeiten Sie zwei Standarddatensätze.

1. Iris: stratifizierter Split, `DummyClassifier` und `LogisticRegression`, Genauigkeitsvergleich.
2. Diabetes: zufälliger Split, `DummyRegressor(strategy="mean")` und `LinearRegression`, MAE-Vergleich.
3. Geben Sie vor und nach `fit` ausgewählte Parameter und gelernte Attribute aus.
4. Erklären Sie den Unterschied zwischen `get_params()` und Attributen mit abschließendem Unterstrich.

> **Hinweis:** Versuchen Sie gedanklich, welche Attribute vor `fit` noch nicht existieren können.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Estimator-API an Klassifikation und Regression prüfen
#
# Ziel dieser Codezelle:
# Bearbeiten Sie zwei Standarddatensätze. 1. Iris: stratifizierter Split,
# DummyClassifier und LogisticRegression, Genauigkeitsvergleich. 2. Diabetes:
# zufälliger Split, DummyRegressor(strategy="mean") und LinearRegressio...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# -----------------------------
# Klassifikation mit Iris
# -----------------------------
iris = load_iris()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    iris.data,
    iris.target,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=iris.target,
)

dummy_classifier = DummyClassifier(strategy="most_frequent")
logistic_classifier = LogisticRegression(max_iter=1000)

# get_params beschreibt Einstellungen, die bereits vor fit existieren.
print("LogReg-Parameter C vor fit:", logistic_classifier.get_params()["C"])

dummy_classifier.fit(X_train_c, y_train_c)
logistic_classifier.fit(X_train_c, y_train_c)

classification_results = pd.DataFrame(
    {
        "model": ["Dummy", "Logistische Regression"],
        "accuracy": [
            accuracy_score(y_test_c, dummy_classifier.predict(X_test_c)),
            accuracy_score(y_test_c, logistic_classifier.predict(X_test_c)),
        ],
    }
)

# coef_ und classes_ werden erst während fit aus Trainingsdaten gelernt.
print("Gelernte Klassen:", logistic_classifier.classes_)
print("Koeffizientenform:", logistic_classifier.coef_.shape)

# -----------------------------
# Regression mit Diabetes
# -----------------------------
diabetes = load_diabetes()
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    diabetes.data,
    diabetes.target,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

dummy_regressor = DummyRegressor(strategy="mean")
linear_regressor = LinearRegression()
dummy_regressor.fit(X_train_r, y_train_r)
linear_regressor.fit(X_train_r, y_train_r)

regression_results = pd.DataFrame(
    {
        "model": ["Mittelwert-Dummy", "Lineare Regression"],
        "MAE": [
            mean_absolute_error(y_test_r, dummy_regressor.predict(X_test_r)),
            mean_absolute_error(y_test_r, linear_regressor.predict(X_test_r)),
        ],
    }
)

print("\nKlassifikation:")
print(classification_results.round(3).to_string(index=False))
print("\nRegression:")
print(regression_results.round(3).to_string(index=False))

### Reflexion zu Aufgabe 1

`get_params()` liefert konfigurierbare Hyperparameter wie `C` oder `max_iter`. Gelernte Attribute wie `coef_`, `intercept_` oder `classes_` entstehen erst durch `fit` und enden nach scikit-learn-Konvention mit einem Unterstrich. `predict` verwendet anschließend diese gelernten Zustände, ohne sie neu zu bestimmen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Numerische und kategoriale Transformatoren einzeln anwenden

    Verwenden Sie `mixed_data` ohne die Zielspalte.

1. Numerisch: Median-Imputation gefolgt von StandardScaler.
2. Kategorial: häufigste Kategorie als Imputation und One-Hot-Encoding mit `handle_unknown="ignore"`.
3. Führen Sie `fit_transform` auf Trainingsdaten aus und `transform` auf Testdaten.
4. Geben Sie Formen, gelernte Medianwerte, Kategorien und erste transformierte Zeilen aus.

> **Hinweis:** Rufen Sie niemals `fit_transform` separat auf dem Testsatz auf.

In [ ]:
X = mixed_data.drop(columns="renewed")
y = mixed_data["renewed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Numerische und kategoriale Transformatoren einzeln anwenden
#
# Ziel dieser Codezelle:
# Verwenden Sie mixeddata ohne die Zielspalte. 1. Numerisch: Median-Imputation
# gefolgt von StandardScaler. 2. Kategorial: häufigste Kategorie als Imputation und
# One-Hot-Encoding mit handleunknown="ignore". 3. Führen Sie...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X = mixed_data.drop(columns="renewed")
y = mixed_data["renewed"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

numeric_columns = ["age", "income_k"]
categorical_columns = ["city", "plan"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

# fit_transform lernt nur aus Training und transformiert es direkt.
X_train_numeric = numeric_transformer.fit_transform(
    X_train[numeric_columns]
)
X_test_numeric = numeric_transformer.transform(
    X_test[numeric_columns]
)

X_train_categorical = categorical_transformer.fit_transform(
    X_train[categorical_columns]
)
X_test_categorical = categorical_transformer.transform(
    X_test[categorical_columns]
)

learned_medians = numeric_transformer.named_steps["imputer"].statistics_
learned_categories = categorical_transformer.named_steps["encoder"].categories_

print("Numerische Formen:", X_train_numeric.shape, X_test_numeric.shape)
print("Kategoriale Formen:", X_train_categorical.shape, X_test_categorical.shape)
print("Gelernte Mediane:", learned_medians)
print("Gelernte Kategorien:", learned_categories)
print("Erste numerische Zeile:", np.round(X_train_numeric[0], 3))
print("Erste kategoriale Zeile:", X_train_categorical[0])

### Reflexion zu Aufgabe 2

Imputer und Skalierer besitzen gelernte Zustände. Sie müssen auf Trainingsdaten gefittet und unverändert auf Testdaten angewendet werden. `handle_unknown="ignore"` verhindert einen Fehler, wenn im Einsatz eine Kategorie erscheint, die beim Fit nicht vorhanden war. Die betreffende One-Hot-Gruppe wird dann zu Nullen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: ColumnTransformer und Merkmalsnamen untersuchen

    Kombinieren Sie die Transformatoren aus Aufgabe 2 in einem `ColumnTransformer`.

1. Fitten Sie ihn auf `X_train`.
2. Transformieren Sie Train und Test.
3. Lesen Sie mit `get_feature_names_out()` die erzeugten Merkmalsnamen aus.
4. Erstellen Sie aus der transformierten Trainingsmatrix einen DataFrame mit diesen Namen.
5. Prüfen Sie, welche Schritte und gelernten Kategorien über `named_transformers_` erreichbar sind.

> **Hinweis:** Die Reihenfolge der Ausgabespalten folgt der Reihenfolge der Transformatoren.

In [ ]:
numeric_columns = ["age", "income_k"]
categorical_columns = ["city", "plan"]

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: ColumnTransformer und Merkmalsnamen untersuchen
#
# Ziel dieser Codezelle:
# Kombinieren Sie die Transformatoren aus Aufgabe 2 in einem ColumnTransformer. 1.
# Fitten Sie ihn auf Xtrain. 2. Transformieren Sie Train und Test. 3. Lesen Sie mit
# getfeaturenamesout() die erzeugten Merkmalsnamen aus....
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

numeric_columns = ["age", "income_k"]
categorical_columns = ["city", "plan"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
transformed_feature_names = preprocessor.get_feature_names_out()

transformed_train_df = pd.DataFrame(
    X_train_transformed,
    columns=transformed_feature_names,
    index=X_train.index,
)

fitted_encoder = (
    preprocessor.named_transformers_["categorical"]
    .named_steps["encoder"]
)

print("Train/Test-Form:", X_train_transformed.shape, X_test_transformed.shape)
print("Merkmalsnamen:", transformed_feature_names.tolist())
print("\nTransformierte Trainingsdaten:")
print(transformed_train_df.head().round(3).to_string())
print("\nKategorien aus dem Encoder:", fitted_encoder.categories_)

### Reflexion zu Aufgabe 3

`ColumnTransformer` wendet unterschiedliche Verarbeitung gezielt auf definierte Spalten an und fügt die Ergebnisse in fester Reihenfolge zusammen. Merkmalsnamen sind wichtig für Debugging und spätere Interpretation. `named_transformers_` enthält die bereits gefitteten Unterobjekte, während `transformers` die ursprüngliche Konfiguration beschreibt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: End-to-End-Pipeline trainieren und Parameter prüfen

    Verbinden Sie den `ColumnTransformer` mit `LogisticRegression` in einer Pipeline.

1. Trainieren Sie die Pipeline direkt mit dem ursprünglichen DataFrame.
2. Berechnen Sie Testgenauigkeit und vorhergesagte Wahrscheinlichkeiten.
3. Lesen Sie verschachtelte Parameter wie `model__C` und `preprocess__numeric__imputer__strategy` aus.
4. Ändern Sie `model__C` mit `set_params`, fitten Sie erneut und vergleichen Sie.
5. Prüfen Sie die Koeffizienten zusammen mit den transformierten Merkmalsnamen.

> **Hinweis:** Fitten Sie nach jeder Hyperparameteränderung erneut.

In [ ]:
# Verwenden Sie den Preprocessor aus Aufgabe 3 oder definieren Sie ihn erneut.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: End-to-End-Pipeline trainieren und Parameter prüfen
#
# Ziel dieser Codezelle:
# Verbinden Sie den ColumnTransformer mit LogisticRegression in einer Pipeline. 1.
# Trainieren Sie die Pipeline direkt mit dem ursprünglichen DataFrame. 2. Berechnen
# Sie Testgenauigkeit und vorhergesagte Wahrscheinlichke...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Eine neue Pipeline verwendet eine frische Modellinstanz und denselben
# logisch definierten Preprocessor.
full_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                C=1.0,
                max_iter=1000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

full_pipeline.fit(X_train, y_train)
first_predictions = full_pipeline.predict(X_test)
first_probabilities = full_pipeline.predict_proba(X_test)[:, 1]
first_accuracy = accuracy_score(y_test, first_predictions)

# Doppelter Unterstrich navigiert durch verschachtelte Pipeline-Schritte.
parameters = full_pipeline.get_params()
print("model__C:", parameters["model__C"])
print(
    "Imputationsstrategie:",
    parameters["preprocess__numeric__imputer__strategy"],
)

# set_params aktualisiert die Konfiguration. Danach muss erneut gefittet
# werden, weil C die Optimierung und Koeffizienten beeinflusst.
full_pipeline.set_params(model__C=0.1)
full_pipeline.fit(X_train, y_train)
second_predictions = full_pipeline.predict(X_test)
second_accuracy = accuracy_score(y_test, second_predictions)

fitted_preprocessor = full_pipeline.named_steps["preprocess"]
feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = full_pipeline.named_steps["model"].coef_[0]
coefficient_table = (
    pd.DataFrame(
        {"feature": feature_names, "coefficient": coefficients}
    )
    .assign(abs_coefficient=lambda df: df["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

print("Erste Genauigkeit:", round(first_accuracy, 3))
print("Zweite Genauigkeit mit C=0.1:", round(second_accuracy, 3))
print("Erste Wahrscheinlichkeiten:", np.round(first_probabilities, 3))
print("\nKoeffizienten:")
print(coefficient_table.round(3).to_string(index=False))

### Reflexion zu Aufgabe 4

Die Pipeline verhindert, dass der Testdatensatz Imputation, Skalierung oder Kategorien beeinflusst. Verschachtelte Parameternamen erlauben spätere Suche und reproduzierbare Konfiguration. Koeffizienten beziehen sich auf die transformierten Merkmale. Wegen der sehr kleinen Beispieltabelle sind ihre Rangfolge und Testgenauigkeit nicht stabil genug für fachliche Schlussfolgerungen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Unbekannte Kategorien und Pipeline-Diagnose

    Erstellen Sie zwei neue Fälle, darunter eine bisher unbekannte Stadt `Leipzig` und einen fehlenden Einkommenswert.

1. Lassen Sie die vollständige Pipeline Wahrscheinlichkeiten vorhersagen.
2. Transformieren Sie die Fälle separat mit dem gefitteten Preprocessor und zeigen Sie die Feature-Matrix mit Namen.
3. Prüfen Sie, welche One-Hot-Spalten für die unbekannte Stadt gesetzt werden.
4. Erstellen Sie eine kleine Diagnose-Tabelle mit Eingaben, Wahrscheinlichkeit und Entscheidung.
5. Erklären Sie, warum fehlerfreier Code noch keine verlässliche Vorhersage für eine unbekannte Gruppe garantiert.

> **Hinweis:** Technische Robustheit gegenüber unbekannten Kategorien ist nicht gleich fachliche Verlässlichkeit.

In [ ]:
new_cases = pd.DataFrame(
    {
        "age": [36, 50],
        "income_k": [np.nan, 92],
        "city": ["Leipzig", "Hamburg"],
        "plan": ["Plus", "Premium"],
    }
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Unbekannte Kategorien und Pipeline-Diagnose
#
# Ziel dieser Codezelle:
# Erstellen Sie zwei neue Fälle, darunter eine bisher unbekannte Stadt Leipzig und
# einen fehlenden Einkommenswert. 1. Lassen Sie die vollständige Pipeline
# Wahrscheinlichkeiten vorhersagen. 2. Transformieren Sie die Fäll...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

new_cases = pd.DataFrame(
    {
        "age": [36, 50],
        "income_k": [np.nan, 92],
        "city": ["Leipzig", "Hamburg"],
        "plan": ["Plus", "Premium"],
    }
)

# Die Pipeline verwendet ihre bereits gelernten Imputationswerte,
# Skalierungsparameter und Kategorien.
new_probabilities = full_pipeline.predict_proba(new_cases)[:, 1]
new_predictions = full_pipeline.predict(new_cases)

fitted_preprocessor = full_pipeline.named_steps["preprocess"]
new_transformed = fitted_preprocessor.transform(new_cases)
feature_names = fitted_preprocessor.get_feature_names_out()
transformed_new_df = pd.DataFrame(
    new_transformed,
    columns=feature_names,
    index=new_cases.index,
)

diagnosis = new_cases.copy()
diagnosis["renewal_probability"] = new_probabilities
diagnosis["predicted_renewed"] = new_predictions

city_columns = [name for name in feature_names if "city_" in name]

print("Diagnose:")
print(diagnosis.round(3).to_string(index=False))
print("\nTransformierte Fälle:")
print(transformed_new_df.round(3).to_string())
print("\nStadt-One-Hot-Spalten:")
print(transformed_new_df[city_columns].to_string())

### Reflexion zu Aufgabe 5

`handle_unknown="ignore"` ermöglicht eine technische Vorhersage für `Leipzig`, indem keine bekannte Stadtspalte aktiviert wird. Das verhindert einen Laufzeitfehler, liefert aber keine Evidenz dafür, dass das Modell für diese neue Gruppe gut kalibriert oder fair ist. Unbekannte Kategorien sollten protokolliert, überwacht und gegebenenfalls durch neue Trainingsdaten abgedeckt werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.